In [ ]:
# =========================
# 1. Imports
# =========================
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

from transformers import (
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    BertForMaskedLM,
    BertTokenizerFast
)

import kagglehub

In [ ]:
# =========================
# 2. Download dataset
# =========================
path = kagglehub.dataset_download("rtatman/natural-stories-corpus")

print("Dataset path:", path)
print("Files:", os.listdir(path))


100%|██████████| 3.82M/3.82M [00:00<00:00, 76.1MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/rtatman/natural-stories-corpus/versions/1
Files: ['batch1_pro.csv', 'batch2_pro.csv', 'all_stories.tok', 'words.tsv']


In [ ]:
# =========================
# 3. File paths (FIXED)
# =========================
base_path = path

words_path = f"{base_path}/words.tsv"
batch1_path = f"{base_path}/batch1_pro.csv"
batch2_path = f"{base_path}/batch2_pro.csv"

# =========================
# 4. Load words.tsv
# =========================
df_words = pd.read_csv(
    words_path,
    sep="\t",
    names=["token_id", "token"]
)

def parse_token_id(token_id):
    parts = token_id.split(".")
    return int(parts[0]), int(parts[1]), parts[2]

df_words[["story_id", "position", "token_type"]] = df_words["token_id"].apply(
    lambda x: pd.Series(parse_token_id(x))
)

# Keep only actual words
df_words_clean = df_words[df_words["token_type"] == "word"].copy()
df_words_clean = df_words_clean.sort_values(["story_id", "position"])
df_words_clean.reset_index(drop=True, inplace=True)

# =========================
# 5. Load RT data
# =========================
batch1 = pd.read_csv(batch1_path)
batch2 = pd.read_csv(batch2_path)

df_rt = pd.concat([batch1, batch2], ignore_index=True)

df_rt = df_rt.rename(columns={
    "WorkerId": "subject",
    "item": "story_id",
    "zone": "position",
    "RT": "rt"
})

# =========================
# 6. Merge
# =========================
df_all = df_words_clean.merge(
    df_rt,
    on=["story_id", "position"]
)

print("Total merged rows:", len(df_all))

df = df_all[df_all["story_id"] == 1].copy()

print("Subset rows:", len(df))

Total merged rows: 1013377
Subset rows: 107300


In [ ]:
# =========================
# 7. Device
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =========================
# 8. Load models
# =========================
# GPT-2
gpt2_tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2")
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token

gpt2_model.to(device)
gpt2_model.eval()

# BERT
bert_tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
bert_model = BertForMaskedLM.from_pretrained("bert-base-uncased")

bert_model.to(device)
bert_model.eval()


Using device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

In [ ]:
# =========================
# 9. GPT-2 (chunked)
# =========================
def compute_gpt2_surprisal_chunked(words, max_length=1024, stride=512):
    text = " ".join(words)

    enc = gpt2_tokenizer(text, return_tensors="pt")
    input_ids = enc["input_ids"][0]

    all_token_surprisal = np.zeros(len(input_ids) - 1)

    start = 0

    while start < len(input_ids):
        end = min(start + max_length, len(input_ids))

        chunk_ids = input_ids[start:end].unsqueeze(0).to(device)

        with torch.no_grad():
            logits = gpt2_model(chunk_ids).logits

        shift_logits = logits[:, :-1, :]
        shift_labels = chunk_ids[:, 1:]

        log_probs = F.log_softmax(shift_logits, dim=-1)

        token_log_probs = log_probs.gather(
            2, shift_labels.unsqueeze(-1)
        ).squeeze(-1)

        surprisal = -token_log_probs.squeeze().cpu().numpy()

        all_token_surprisal[start:start+len(surprisal)] = surprisal

        if end == len(input_ids):
            break

        start += stride

    # Aggregate to words
    encoding = gpt2_tokenizer(text)
    word_ids = encoding.word_ids()

    word_surprisals = []
    current_word = None
    current_sum = 0.0

    for idx, word_id in enumerate(word_ids[1:]):
        if word_id is None:
            continue

        if word_id != current_word:
            if current_word is not None:
                word_surprisals.append(current_sum)
            current_word = word_id
            current_sum = 0.0

        current_sum += all_token_surprisal[idx]

    if current_word is not None:
        word_surprisals.append(current_sum)

    return word_surprisals

In [ ]:
# =========================
# 10. BERT (chunked)
# =========================
def compute_bert_surprisal(words):
    text = " ".join(words)

    encoding = bert_tokenizer(text, return_tensors="pt")
    input_ids = encoding["input_ids"][0].to(device)
    word_ids = encoding.word_ids()

    word_surprisals = []

    unique_word_ids = sorted(set([w for w in word_ids if w is not None]))

    for word_id in unique_word_ids:
        masked_input = input_ids.clone()

        token_indices = [i for i, w in enumerate(word_ids) if w == word_id]

        for idx in token_indices:
            masked_input[idx] = bert_tokenizer.mask_token_id

        masked_input = masked_input.unsqueeze(0)

        with torch.no_grad():
            logits = bert_model(masked_input).logits

        log_probs = F.log_softmax(logits, dim=-1)

        word_log_prob = 0.0
        for idx in token_indices:
            true_id = input_ids[idx]
            word_log_prob += log_probs[0, idx, true_id].item()

        word_surprisals.append(-word_log_prob)

    return word_surprisals


def compute_bert_surprisal_chunked(words, max_length=512):
    all_surprisals = []

    for i in range(0, len(words), max_length):
        chunk = words[i:i + max_length]
        all_surprisals.extend(compute_bert_surprisal(chunk))

    return all_surprisals


Kiarash Task :

Gabe Task :  validation testing